# SageMaker endpointでテキストEmbeddingを生成

`project_name`、`project_objective`、`project_summary`をラベル付きテキストへ整形し、Amazon SageMaker上のモデルをreal-time endpoint経由で呼び出します。生成物はすべて`data/embeddings/`以下に保存します。

安全のため、このNotebookは初期状態ではdry-runしか行いません。まず入力件数とtruncate候補を確認し、次に5行のsmoke test、最後に全件実行の順で進めます。`science_tech_decision`はendpointへ送信しません。

In [ ]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from embedding_features import (
    build_sagemaker_json_request,
    generate_embeddings,
    l2_normalize_embeddings,
    load_embeddings,
    parse_sagemaker_json_response,
)

## 設定

SageMaker上ではexecution role、ローカルではAWS profileなど、boto3の標準credential chainを使います。Access keyをNotebookへ直接書かないでください。endpoint名は`SAGEMAKER_ENDPOINT_NAME`で渡せます。

endpointのrequest/response仕様はまだ確定していないため、`REQUEST_BUILDER`と`RESPONSE_PARSER`を独立させています。標準設定は`{"inputs": [...]}`を送り、一般的なJSON embedding形式を読み取ります。仕様決定後はこの2関数だけ差し替えてください。`ADAPTER_ID`も更新すると古いcacheとの混同を防げます。

In [ ]:
PROVIDER = 'sagemaker'
MODEL = 'jp-embedding-v1'  # endpoint内モデルのcache識別名
ENDPOINT_NAME = os.getenv('SAGEMAKER_ENDPOINT_NAME')
REGION_NAME = os.getenv('AWS_REGION') or os.getenv('AWS_DEFAULT_REGION') or 'ap-northeast-1'
AWS_PROFILE_NAME = os.getenv('AWS_PROFILE')  # SageMaker上では通常None
EMBEDDING_DIM = None  # 確定後に例: 768。Noneなら初回responseから解決
CONTENT_TYPE = 'application/json'
ACCEPT = 'application/json'
REQUEST_BUILDER = build_sagemaker_json_request
RESPONSE_PARSER = parse_sagemaker_json_response
ADAPTER_ID = 'json-inputs-v1'  # payload/parser変更時に必ず更新
INVOKE_ENDPOINT_KWARGS = {}  # 例: {'TargetVariant': 'variant-b'}
SAGEMAKER_CACHE_IDENTITY = {}  # 上記kwargsが出力を変える場合に同じ情報を記録

TEXT_COLS = ['project_name', 'project_objective', 'project_summary']
TARGET_COL = 'science_tech_decision'
OUTPUT_ROOT = PROJECT_ROOT / 'data' / 'embeddings'
BATCH_SIZE = 32
PRICE_PER_MILLION_TOKENS = 0.0  # SageMakerは通常instance稼働時間で課金
MAX_BUDGET_USD = 0.0  # token課金guardは無効。endpoint料金は別途確認
MAX_INPUT_TOKENS = 8000

RUN_SMOKE_API = False  # 5行だけ実APIを呼ぶときTrue
RUN_FULL_API = False   # 全件を実APIで生成するときTrue

In [ ]:
train = pd.read_csv(PROJECT_ROOT / 'input' / 'train.csv')
test = pd.read_csv(PROJECT_ROOT / 'input' / 'test.csv')

display(train[TEXT_COLS].head())
print('train:', train.shape, 'test:', test.shape)

## 1. dry-run（API呼び出しなし）

行数、空テキスト、truncate候補、推定token数を確認します。ここではboto3 clientを作成せず、endpointも呼び出しません。

In [ ]:
common_kwargs = dict(
    provider=PROVIDER,
    model=MODEL,
    output_root=OUTPUT_ROOT,
    text_cols=TEXT_COLS,
    target_col=TARGET_COL,
    embedding_dim=EMBEDDING_DIM,
    batch_size=BATCH_SIZE,
    max_input_tokens=MAX_INPUT_TOKENS,
    price_per_million_tokens=PRICE_PER_MILLION_TOKENS,
    max_budget_usd=MAX_BUDGET_USD,
    endpoint_name=ENDPOINT_NAME,
    region_name=REGION_NAME,
    aws_profile_name=AWS_PROFILE_NAME,
    content_type=CONTENT_TYPE,
    accept=ACCEPT,
    request_builder=REQUEST_BUILDER,
    response_parser=RESPONSE_PARSER,
    invoke_endpoint_kwargs=INVOKE_ENDPOINT_KWARGS,
    adapter_id=ADAPTER_ID,
    sagemaker_cache_identity=SAGEMAKER_CACHE_IDENTITY,
)

train_dry_run = generate_embeddings(train, split='train', dry_run=True, **common_kwargs)
test_dry_run = generate_embeddings(test, split='test', dry_run=True, **common_kwargs)
dry_run_reports = pd.DataFrame([train_dry_run['report'], test_dry_run['report']])
display(dry_run_reports)
print('train + test rows:', int(dry_run_reports['total_rows'].sum()))
print('train + test estimated tokens:', int(dry_run_reports['total_estimated_tokens'].sum()))
TOTAL_ESTIMATED_COST_USD = float(dry_run_reports['estimated_cost_usd'].sum())
print('token-based estimated cost (USD):', TOTAL_ESTIMATED_COST_USD)
print('Note: SageMaker endpoint cost is based on deployed instance uptime.')

## API contractを差し替える場合

payload仕様が`{"inputs": [...]}`でない場合は、下の形でbuilder/parserを定義し、設定セルの`REQUEST_BUILDER`と`RESPONSE_PARSER`へ代入します。parserにはresponse bodyの`bytes`と期待行数が渡されます。変更時は`ADAPTER_ID`も`v2`などへ更新してください。

In [ ]:
# def custom_request_builder(texts, model, embedding_dim):
#     return {'sentences': list(texts), 'parameters': {'dimension': embedding_dim}}
#
# def custom_response_parser(body, expected_rows):
#     import json
#     payload = json.loads(body.decode('utf-8'))
#     matrix = np.asarray(payload['result'], dtype=np.float32)
#     assert matrix.shape[0] == expected_rows
#     return matrix
#
# REQUEST_BUILDER = custom_request_builder
# RESPONSE_PARSER = custom_response_parser
# ADAPTER_ID = 'custom-contract-v2'

def current_runtime_kwargs():
    # このセルでadapterを変更した後でもsmoke/full実行へ反映する。
    return {
        **common_kwargs,
        'request_builder': REQUEST_BUILDER,
        'response_parser': RESPONSE_PARSER,
        'adapter_id': ADAPTER_ID,
    }

## 2. 5行のsmoke test

dry-runとendpoint設定を確認した後、`RUN_SMOKE_API=True`にして実行します。保存先は本番と分離した`data/embeddings_smoke/`です。

In [ ]:
smoke_result = None
if RUN_SMOKE_API:
    if not ENDPOINT_NAME:
        raise ValueError('Set SAGEMAKER_ENDPOINT_NAME or ENDPOINT_NAME before the smoke test.')
    smoke_kwargs = {**current_runtime_kwargs(), 'output_root': PROJECT_ROOT / 'data' / 'embeddings_smoke'}
    smoke_result = generate_embeddings(
        train.head(5),
        split='train_smoke',
        dry_run=False,
        **smoke_kwargs,
    )
    assert smoke_result['embeddings'].shape[0] == min(5, len(train))
    assert np.isfinite(smoke_result['embeddings']).all()
    display(smoke_result['metadata'])
else:
    print('Smoke API call is disabled. Set RUN_SMOKE_API=True after checking the dry-run.')

## 3. train/test全件生成

`RUN_FULL_API=True`の場合だけSageMaker endpointを呼びます。batchごとにshardと進捗を保存するため、停止後に同じ設定で再実行すると完了済みbatchを検証して再利用します。完成済みcacheがあればAWS credentialなしでも読み込めます。endpointのinstance料金はtoken数ではなく稼働時間を基準に別途確認してください。

In [ ]:
train_result = None
test_result = None
if RUN_FULL_API:
    if not ENDPOINT_NAME:
        raise ValueError('Set SAGEMAKER_ENDPOINT_NAME or ENDPOINT_NAME before the full run.')
    runtime_kwargs = current_runtime_kwargs()
    train_result = generate_embeddings(train, split='train', dry_run=False, **runtime_kwargs)
    test_result = generate_embeddings(test, split='test', dry_run=False, **runtime_kwargs)
    print('train:', train_result['embeddings'].shape, train_result['cache_dir'])
    print('test :', test_result['embeddings'].shape, test_result['cache_dir'])
else:
    print('Full API call is disabled. Set RUN_FULL_API=True only after the smoke test succeeds.')

## 読み込み・整合性確認・L2正規化

`generate_embeddings`の戻り値には保存先が含まれます。`load_embeddings`へ元DataFrameを渡すと、行数だけでなく`project_id`と元indexの順序も検証します。保存するのはraw embeddingで、正規化が必要なモデルだけ別途コピーを作ります。

In [ ]:
if train_result is not None:
    train_embeddings, train_metadata = load_embeddings(
        train_result['cache_dir'],
        split='train',
        expected_df=train,
    )
    train_embeddings_l2 = l2_normalize_embeddings(train_embeddings)
    assert train_embeddings.shape[0] == len(train_metadata) == len(train)
    assert np.isfinite(train_embeddings_l2).all()
    print(train_embeddings.shape, train_embeddings.dtype)